# Resultados de la evaluación de M0–M3

Este notebook **no vuelve a correr la evaluación**: lee lo que `src.eval.run_eval` dejó en `reports/` y genera tablas, figuras y comparaciones listas para el informe y la exposición.

| Qué produce | Dónde queda |
|---|---|
| Figuras para diapositivas (PNG 300 dpi + PDF, en inglés) | `reports/figures/<split>/` |
| Tablas para el informe (CSV + LaTeX) | `reports/tables/<split>/` |
| Predicciones fuera de pliegue de M1 y M2 (solo train) | `reports/eval_train/oof/` |

**Cómo usarlo**
1. Deja `SPLIT = "train"` mientras se desarrolla. Cuando exista la corrida final sobre el *test*, cambia a `SPLIT = "test"` y ejecuta todo de nuevo: no hay que tocar nada más.
2. Ejecuta de arriba hacia abajo desde la carpeta `notebooks/` con el entorno virtual del proyecto.
3. Las métricas oficiales son las de `results_<fecha>.csv`; este notebook primero comprueba que las puede reproducir desde las predicciones por pregunta (`manual_eval_<fecha>.csv`).

In [ ]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # las figuras se guardan en disco; abajo se muestran con display
import matplotlib.pyplot as plt
import pandas as pd

try:
    from IPython.display import Image, display
except ImportError:  # ejecución fuera de Jupyter
    display = print

    def Image(filename: str) -> str:  # noqa: N802 - mismo nombre que en IPython
        return filename

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import ESCALATING_INTENTS, INTENTS, PROCESSED_DIR, REPORTS_DIR, SEED
from src.eval import visualize_results as vr

# ---- Configuración ----------------------------------------------------------------
SPLIT = "train"  # "train" (desarrollo) o "test" (resultados finales, una sola corrida)
EVAL_DIRS = {
    "train": [REPORTS_DIR / "eval_train" / "clasicos", REPORTS_DIR / "eval_train" / "m3"],
    "test": [REPORTS_DIR / "eval_test" / "clasicos", REPORTS_DIR / "eval_test" / "m3"],
}
RUN_OOF = True  # validación cruzada de M1 y M2 sobre train (necesita los motores)
N_BOOT = 2000
LABELED_PATH = PROCESSED_DIR / "labeled.csv"
FIG_DIR = REPORTS_DIR / "figures" / SPLIT
TAB_DIR = REPORTS_DIR / "tables" / SPLIT
OOF_DIR = REPORTS_DIR / "eval_train" / "oof"
SPLIT_EN = {"train": "train split (development)", "test": "test split (final)"}[SPLIT]

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print(f"Split: {SPLIT} | semilla: {SEED} | bootstrap: {N_BOOT}")
print("Carpetas:", *[str(p.relative_to(ROOT)) for p in EVAL_DIRS[SPLIT]], sep="\n  ")


def show(path_base: Path) -> None:
    # Muestra el PNG guardado (lo que irá a la diapositiva).
    display(Image(filename=str(Path(path_base).with_suffix(".png"))))

## 1. Los datos evaluados

`labeled.csv` (columnas `intent`, `should_escalate`, `split`). Las intenciones marcadas `escala = True` deben pasar a una persona. Las respondibles son las únicas que tienen fragmento en la base (`kb_source`).

In [ ]:
try:
    from src.eval.run_eval import load_datasets

    labeled = load_datasets([LABELED_PATH])
except ImportError:  # el kit también funciona sin el arnés
    labeled = pd.read_csv(LABELED_PATH, dtype={"question_id": str}, keep_default_na=False)

overview = vr.dataset_overview(labeled)
display(overview)
vr.save_table(
    overview, TAB_DIR / "tabla_datos", "Preguntas reales por intención y partición", "tab:datos"
)

## 2. Carga y verificación

Se unen las carpetas de los motores clásicos y de M3. Antes de analizar nada se comprueba que:
- todos los motores evaluaron **las mismas preguntas** (si no, `load_eval_dirs` falla);
- las métricas recalculadas desde `manual_eval` coinciden con `results_<fecha>.csv` (columna `coincide`);
- las matrices de confusión reconstruidas coinciden con los `confusion_matrix_*.csv`.

In [ ]:
run = vr.load_eval_dirs(EVAL_DIRS[SPLIT], split=SPLIT)
outcomes = vr.per_question_outcomes(run.predictions)
metrics = vr.compute_metrics(outcomes, n_boot=N_BOOT)

check = vr.check_against_summary(metrics, run.summary)
display(check)
display(vr.check_confusion(run))
assert check["coincide"].all(), "Las métricas recalculadas no coinciden con results_*.csv"
print("Motores:", run.engines)
print("Archivos leídos:")
for kind, paths in run.sources.items():
    for path in paths:
        print(f"  [{kind}] {Path(path).relative_to(ROOT)}")

## 3. Qué métrica aplica a qué motor (y por qué)

| Métrica | M0 | M1 | M2 | M3 | Por qué |
|---|---|---|---|---|---|
| Accuracy de intención (todas) | sí | sí* | no aplica | no aplica | M2/M3 toman la intención del fragmento top-1 y la base solo tiene fragmentos de `capacidad`, `reglas_servicios` y `ubicacion`. Nunca pueden predecir las 4 intenciones que escalan: su techo es n_respondibles / n_total. Se reporta la accuracy **solo sobre respondibles**. |
| Hit@k | no aplica | no aplica | sí | sí (= M2) | M0/M1 no recuperan fragmentos. M3 usa el recuperador de M2: mismos `retrieved_str`, mismo Hit@k y misma matriz de confusión. |
| Abstención correcta / innecesaria | sí | sí* | sí | sí | Aplica a todos: es la decisión escalar/responder. |
| Corrección | automática | automática* | automática | **manual** | M0/M1 devuelven la plantilla validada de la intención y M2 el fragmento sin redactar: la respuesta es correcta si no escaló y acertó la intención (M0/M1) o el fragmento top-1 (M2). M3 redacta texto libre: se califica a mano. |
| Alucinación | 0 por construcción | 0 por construcción | 0 por construcción | **manual** | M0–M2 solo eligen entre textos ya validados; no pueden afirmar algo que no esté en la base. |
| Latencia | sí | sí | sí | sí | Todos medidos en el mismo equipo (columna `hardware`). |

\* **M1 sobre train no es válido**: se evalúa con las mismas filas con las que se entrenó (resustitución) y las memoriza. Para train se usa la validación cruzada fuera de pliegue de la sección 9; sobre test sí es válido.

In [ ]:
MAIN_METRICS = [
    "intent_accuracy",
    "intent_accuracy_respondable",
    "correct_abstention",
    "unnecessary_abstention",
    "escalation_decision_accuracy",
    "youden_j",
    "auto_correct_respondable",
    "hit_at_1",
    "hit_at_3",
    "hit_at_5",
]
main_table = vr.wide_metric_table(metrics, MAIN_METRICS)
# La accuracy global de M2/M3 se muestra, pero se marca: no es comparable con M0/M1.
for engine in ("M2", "M3"):
    if engine in main_table.index:
        col = vr.METRIC_LABELS_ES["intent_accuracy"]
        main_table.loc[engine, col] = main_table.loc[engine, col] + " (techo por diseño)"
if SPLIT == "train" and "M1" in main_table.index:
    main_table.index = [f"{e} (resustitución)" if e == "M1" else e for e in main_table.index]
sizes = run.summary.set_index("engine")[["n_total", "n_should_escalate", "n_respondable", "n_kb"]]
print("Tamaños (results_*.csv: n_total, n_should_escalate, n_respondable, n_kb):")
display(sizes)
display(main_table)
vr.save_table(
    main_table,
    TAB_DIR / "tabla_metricas",
    f"Métricas por motor ({SPLIT}); punto [IC 95 % bootstrap]",
    f"tab:metricas_{SPLIT}",
)

## 4. Matrices de confusión

Filas = intención real, columnas = intención predicha. El **color es el % de la fila** (recall), para que `disponibilidad` y `otro_negocio`, que son las más frecuentes, no opaquen al resto; cada celda muestra el conteo y el %. `(E)` marca las intenciones que deben escalar. Se dibuja una sola matriz para M2/M3 porque es idéntica.

In [ ]:
titles = {
    "M0": f"M0 Keyword rules · {SPLIT}",
    "M1": f"M1 TF-IDF + LR · {SPLIT}" + (" (resubstitution, not valid)" if SPLIT == "train" else ""),
    "M2": f"M2 / M3 retriever (top-1 fragment intent) · {SPLIT}",
}
for engine in ("M0", "M1", "M2"):
    if engine in run.confusion:
        base = FIG_DIR / f"confusion_{engine}"
        vr.plot_confusion(run.confusion[engine], titles[engine], base)
        show(base)
grid = {e: run.confusion[e] for e in ("M0", "M1", "M2") if e in run.confusion}
vr.plot_confusion_grid(grid, {"M0": "M0 rules", "M1": titles["M1"].split(" · ")[0]
                              + (" (resubstitution)" if SPLIT == "train" else ""),
                              "M2": "M2 / M3 retriever"}, FIG_DIR / "confusion_grid")

## 5. La decisión escalar/responder

Es la figura principal para un agente que debe escalar. Panel izquierdo: de las preguntas que **debían escalar**, cuántas escaló (verde) y cuántas respondió (rojo: riesgo de dar información no validada, p. ej. una fecha o un precio). Panel derecho: de las **respondibles**, cuántas respondió bien, mal, escaló sin necesidad, o quedan para calificación manual (M3).

Origen: columnas `should_escalate`, `escalate`, `predicted_intent`, `retrieved_str` y `kb_source` de `manual_eval_<fecha>.csv`.

In [ ]:
esc_table = vr.escalation_outcomes(outcomes)
display(esc_table)
labels = {"M1": "M1 TF-IDF + LR (resubst.)"} if SPLIT == "train" else {}
vr.plot_escalation_outcomes(esc_table, FIG_DIR / "escalation_outcomes", engine_labels=labels,
                            title=f"What each engine does with each question · {SPLIT_EN}")
show(FIG_DIR / "escalation_outcomes")
vr.save_table(esc_table.set_index("engine"), TAB_DIR / "tabla_escalar_responder",
              f"Resultado por pregunta ({SPLIT})", f"tab:escalar_{SPLIT}")

## 6. Métricas con intervalos de confianza

Barras = valor puntual; líneas = IC 95 % bootstrap (2000 remuestreos, semilla 42). Si los IC de dos motores se solapan mucho, mira la sección 8 (diferencias pareadas) antes de concluir.

**Ojo con el "acierto en la decisión escalar/responder"**: en estos datos la gran mayoría de preguntas debe escalar, así que un motor que **siempre** escala ya acierta esa proporción (n_should_escalate / n_total) sin servir para nada. Por eso se reporta también **J = abstención correcta − abstención innecesaria**: vale 0 para "siempre escalar" y 1 para un motor perfecto. Es el mismo criterio con el que M1 y M2 eligen su umbral.

In [ ]:
na = {(e, "intent_accuracy"): "n/a\nby design" for e in ("M2", "M3")}
na[("M3", "auto_correct_respondable")] = "manual\ngrading"
vr.plot_metric_panels(
    metrics,
    ["intent_accuracy", "correct_abstention", "unnecessary_abstention", "youden_j",
     "auto_correct_respondable"],
    FIG_DIR / "metrics_panels",
    engine_labels=labels,
    na_notes=na,
    title=f"Engine comparison · {SPLIT_EN}",
)
show(FIG_DIR / "metrics_panels")

vr.plot_hit_at_k(metrics, "M2", FIG_DIR / "hit_at_k")
show(FIG_DIR / "hit_at_k")

vr.plot_latency(run.summary, FIG_DIR / "latency")
show(FIG_DIR / "latency")
lat = run.summary.set_index("engine")[["latency_median_ms", "latency_p95_ms", "hardware"]]
display(lat)
vr.save_table(lat.round(2), TAB_DIR / "tabla_latencia", "Latencia por respuesta (ms)",
              f"tab:latencia_{SPLIT}")

## 7. ¿Dónde se equivoca cada motor? (por intención)

`recall` = proporción de preguntas de esa intención con la intención correcta; `escalo` = proporción que escaló. En M2/M3 el recall de las intenciones que escalan es 0 por diseño.

In [ ]:
intent_table = vr.per_intent_table(outcomes)
display(intent_table.pivot(index="intent", columns="engine", values=["recall", "escalo"]).round(2))
vr.plot_per_intent_recall(intent_table, FIG_DIR / "intent_recall",
                          title=f"Intent recall per intent · {SPLIT_EN}")
show(FIG_DIR / "intent_recall")
vr.save_table(intent_table.round(3), TAB_DIR / "tabla_por_intencion",
              f"Recall y escalamiento por intención ({SPLIT})", f"tab:intencion_{SPLIT}",
              index=False)

## 8. Diferencias pareadas entre motores

Como todos los motores respondieron **las mismas preguntas**, la comparación correcta es pareada: se remuestrean preguntas y se mide la diferencia `a − b` en cada remuestreo. Si el IC incluye 0 (`ic_incluye_0 = True`), con este tamaño de muestra la diferencia no se distingue del azar.

In [ ]:
comparisons = vr.paired_comparisons(outcomes, n_boot=N_BOOT)
display(comparisons.round(3))
vr.save_table(comparisons.round(3), TAB_DIR / "tabla_comparaciones_pareadas",
              f"Diferencias pareadas (a - b) con IC 95 % ({SPLIT})", f"tab:pareadas_{SPLIT}",
              index=False)

## 9. M1 y M2 honestos sobre train: validación cruzada fuera de pliegue

Solo en `train`. Se parte el train en 5 pliegues estratificados por intención (semilla 42). Cada pliegue se predice con un motor entrenado **sin** esas preguntas, usando únicamente `fit()` y `respond()`. Lo que M1 elige por dentro (C y umbral, con su propia validación cruzada) se repite en cada pliegue: es una validación cruzada anidada. El test no participa.

También se guarda lo que elige `fit()` con **todo** el train: es la configuración final de M1 y M2 que se usa sobre el test y conviene documentarla en el informe.

Las predicciones se guardan en `reports/eval_train/oof/` y, si ya existen, se leen en vez de recalcularse (borra la carpeta para repetir).

In [ ]:
oof = None
try:
    import src.engines.m1_tfidf_rl  # noqa: F401
    import src.engines.m2_embeddings  # noqa: F401
except ImportError as exc:
    if RUN_OOF:
        print(f"No se pudieron importar los motores ({exc}); se omite esta sección.")
    RUN_OOF = False

if SPLIT == "train" and RUN_OOF:
    oof_preds_path = OOF_DIR / "oof_predictions.csv"
    if oof_preds_path.exists():
        oof = pd.read_csv(oof_preds_path, dtype={"question_id": str})
        folds_table = pd.read_csv(OOF_DIR / "oof_folds.csv")
        final_choices = pd.read_csv(OOF_DIR / "final_fit_choices.csv")
        print(f"Leído de {oof_preds_path.relative_to(ROOT)}")
    else:
        from src.engines.m1_tfidf_rl import TfidfRlEngine
        from src.engines.m2_embeddings import EmbeddingEngine, SentenceTransformerEncoder

        # Tarda unos minutos: M1 repite su validación cruzada interna en cada pliegue.
        train_df = labeled[labeled["split"].astype(str).str.strip() == "train"].copy()
        encoder = SentenceTransformerEncoder()  # se carga una vez y se reutiliza
        oof_m1, folds_m1 = vr.out_of_fold_predictions(TfidfRlEngine, train_df)
        oof_m2, folds_m2 = vr.out_of_fold_predictions(
            lambda: EmbeddingEngine(encoder=encoder), train_df
        )
        oof = pd.concat([oof_m1, oof_m2], ignore_index=True)
        folds_table = pd.concat([folds_m1, folds_m2], ignore_index=True)

        final_m1 = TfidfRlEngine()
        final_m1.fit(train_df)
        final_m2 = EmbeddingEngine(encoder=encoder)
        final_m2.fit(train_df)
        final_choices = pd.DataFrame([vr.describe_engine(final_m1), vr.describe_engine(final_m2)])
        tradeoffs = {
            "M1": final_m1.get_threshold_tradeoff_table(),
            "M2": final_m2.get_threshold_tradeoff_table(),
        }
        OOF_DIR.mkdir(parents=True, exist_ok=True)
        oof.to_csv(oof_preds_path, index=False, encoding="utf-8")
        folds_table.to_csv(OOF_DIR / "oof_folds.csv", index=False, encoding="utf-8")
        final_choices.to_csv(OOF_DIR / "final_fit_choices.csv", index=False, encoding="utf-8")
        for name, table in tradeoffs.items():
            table.to_csv(OOF_DIR / f"threshold_tradeoff_{name}.csv", index=False, encoding="utf-8")

    print("Configuración final elegida por fit() con todo el train (documentar en el informe):")
    display(final_choices)
    print("Lo que eligió cada pliegue (estabilidad de la elección):")
    display(folds_table)
else:
    print("Sección omitida (solo aplica a train con RUN_OOF = True).")

In [ ]:
honest = None
if oof is not None:
    # Vista honesta de train: M0 (no se entrena), M1 y M2 fuera de pliegue, M3 de la corrida.
    oof_ready = oof.copy()
    keep = run.predictions[run.predictions["engine"].isin(["M0", "M3"])]
    honest_preds = pd.concat([keep, oof_ready], ignore_index=True)
    honest = vr.per_question_outcomes(honest_preds)
    honest_metrics = vr.compute_metrics(honest, n_boot=N_BOOT)
    honest_table = vr.wide_metric_table(honest_metrics, MAIN_METRICS)
    honest_table.index = [
        {"M1": "M1 (5-fold OOF)", "M2": "M2 (5-fold OOF)"}.get(e, e) for e in honest_table.index
    ]
    display(honest_table)
    vr.save_table(honest_table, TAB_DIR / "tabla_metricas_oof",
                  "Train: M1 y M2 fuera de pliegue (5 pliegues); punto [IC 95 %]",
                  "tab:metricas_oof")
    oof_labels = {"M1": "M1 TF-IDF + LR (5-fold OOF)", "M2": "M2 Embeddings (5-fold OOF)"}
    vr.plot_escalation_outcomes(vr.escalation_outcomes(honest), FIG_DIR / "escalation_outcomes_oof",
                                engine_labels=oof_labels,
                                title="Train split, honest view (M1 and M2 out-of-fold)")
    show(FIG_DIR / "escalation_outcomes_oof")
    vr.plot_metric_panels(
        honest_metrics,
        ["intent_accuracy", "correct_abstention", "unnecessary_abstention", "youden_j",
         "auto_correct_respondable"],
        FIG_DIR / "metrics_panels_oof",
        engine_labels=oof_labels,
        na_notes=na,
        title="Train split, honest view (M1 and M2 out-of-fold)",
    )
    show(FIG_DIR / "metrics_panels_oof")
    oof_m1_cm = pd.crosstab(oof[oof.engine == "M1"]["intent"], oof[oof.engine == "M1"]["predicted_intent"])
    vr.plot_confusion(oof_m1_cm, "M1 TF-IDF + LR · train, 5-fold out-of-fold", FIG_DIR / "confusion_M1_oof")
    show(FIG_DIR / "confusion_M1_oof")
    display(vr.paired_comparisons(honest, n_boot=N_BOOT).round(3))

## 10. M3: ¿escala cuando dice que va a escalar?

Análisis de sensibilidad (no reemplaza la métrica oficial). Columnas `escalate` y `answer` de `manual_eval` (M3):
- **`'ESCALAR'` en el texto**: el LLM escribió la palabra `ESCALAR` pero el motor no marcó `escalate=True`, porque solo la detecta cuando es toda la respuesta. Es un fallo de lectura de la salida, no del LLM.
- **Remite a una persona (heurística)**: respuestas tipo "comunícate con…" sin `escalate=True`. Se detectan con palabras clave, así que es una **estimación**, no una medida.

También se muestra qué tan a menudo el fragmento correcto llegó al LLM (`kb_source` en el top-3 de `retrieved_str`): es el techo de lo que M3 puede responder bien.

In [ ]:
if "M3" in run.engines:
    sens = vr.m3_token_sensitivity(outcomes)
    display(sens.round(3))
    vr.save_table(sens.round(3), TAB_DIR / "tabla_m3_sensibilidad",
                  f"M3: abstención según cómo se lea su salida ({SPLIT})",
                  f"tab:m3_sens_{SPLIT}", index=False)
    display(vr.wide_metric_table(metrics, ["m3_context_has_source", "m3_token_leak",
                                           "m3_soft_redirect"]).loc[["M3"]])

## 11. Motores combinados (simulación)

El diseño prevé que el bot final pueda usar una combinación, p. ej. un motor que decide si escalar y otro que responde. Aquí se **simula** con las predicciones guardadas: la pregunta escala si escala cualquiera de los dos; si no, responde el segundo. No se ejecuta ningún motor, así que la latencia de la combinación es aproximadamente la suma de ambas.

En train, el motor que decide (M1) debe venir de la validación fuera de pliegue (sección 9); con la corrida de train de M1 la simulación no sería válida y se omite.

In [ ]:
gate_source = None
if SPLIT == "test":
    gate_source = run.predictions
elif oof is not None:
    gate_source = pd.concat([run.predictions[run.predictions.engine != "M1"],
                             oof[oof.engine == "M1"]], ignore_index=True)

COMBO_ORDER = ["M0", "M1", "M2", "M3", "M0+M2", "M0+M3", "M1+M2", "M1+M3"]
if gate_source is None:
    print("Sin predicciones válidas de M1 para simular la combinación en train.")
else:
    combos = []
    for gate in ("M0", "M1"):
        for answerer in ("M2", "M3"):
            if {gate, answerer} <= set(gate_source["engine"]):
                combos.append(vr.simulate_gate(gate_source, gate=gate, answerer=answerer))
    base_engines = gate_source[gate_source.engine.isin(["M0", "M1", "M2", "M3"])]
    combo_preds = pd.concat([base_engines, *combos], ignore_index=True)
    combo_out = vr.per_question_outcomes(combo_preds)
    combo_metrics = vr.compute_metrics(combo_out, n_boot=N_BOOT)
    combo_table = vr.wide_metric_table(
        combo_metrics,
        ["correct_abstention", "unnecessary_abstention", "youden_j",
         "escalation_decision_accuracy", "auto_correct_respondable"],
    )
    combo_table = combo_table.reindex([e for e in COMBO_ORDER if e in combo_table.index])
    if SPLIT == "train":
        combo_table.index = ["M1 (5-fold OOF)" if e == "M1" else e for e in combo_table.index]
    display(combo_table)
    vr.save_table(combo_table, TAB_DIR / "tabla_combinaciones",
                  f"Combinaciones simuladas: un clasificador decide si escalar ({SPLIT})",
                  f"tab:combinaciones_{SPLIT}")
    combo_esc = vr.escalation_outcomes(combo_out)
    order = [e for e in COMBO_ORDER if e in set(combo_esc.engine)]
    combo_esc = combo_esc.set_index("engine").loc[order].reset_index()
    combo_labels = {
        "M0+M2": "M0 gate + M2 answer", "M0+M3": "M0 gate + M3 answer",
        "M1+M2": "M1 gate + M2 answer", "M1+M3": "M1 gate + M3 answer",
    }
    if SPLIT == "train":
        combo_labels["M1"] = "M1 TF-IDF + LR (5-fold OOF)"
    vr.plot_escalation_outcomes(combo_esc, FIG_DIR / "escalation_outcomes_combined",
                                engine_labels=combo_labels,
                                title=f"Single engines vs. simulated combinations · {SPLIT_EN}")
    show(FIG_DIR / "escalation_outcomes_combined")

## 12. Archivos generados

Todo lo que va a la presentación está en `reports/figures/<split>/` (usar los `.png` en diapositivas y los `.pdf` en el artículo). Las tablas `.tex` se pegan en la plantilla IEEE (requieren `\usepackage{booktabs}` y `\usepackage{graphicx}`).

In [ ]:
for folder in (FIG_DIR, TAB_DIR):
    print(folder.relative_to(ROOT))
    for path in sorted(folder.glob("*")):
        print("  ", path.name)